# 03 — SIM, trecho recuperado (NÃO É notebook completo)

O trecho de código abaixo foi recuperado de material anexado. Depende das variáveis e funções declaradas nas Partes 1 e 2 originais, que não estão incluídas aqui. **Não executar isoladamente.**

In [ ]:
# ============================================================
# MEDIA ANALYTICS
# NB 03 - SIM
# PARTE 3/4 - INGESTAO BRONZE
# ============================================================

print("=" * 70)
print("MEDIA ANALYTICS - INGESTAO SIM/DO")
print("=" * 70)

# ============================================================
# 01 - CONTROLE DE ARQUIVOS
# ============================================================

def id_sim(item):

    return hashlib.sha256(
        item["chave"].encode("utf-8")
    ).hexdigest()[:20]


def destino_sim(item):

    pasta = (
        BRONZE
        / f"uf={item['uf']}"
        / f"ano={int(item['ano'])}"
    )

    pasta.mkdir(
        parents=True,
        exist_ok=True
    )

    return pasta / f"{id_sim(item)}.parquet"


def controle_sim(item):

    return CONTROLE / f"{id_sim(item)}.json"


def validar_parquet_sim(caminho):

    caminho = Path(caminho)

    if caminho.is_file():

        if caminho.stat().st_size <= 0:
            raise RuntimeError(
                "Arquivo Parquet vazio."
            )

        return int(
            pq.read_metadata(str(caminho)).num_rows
        )

    if caminho.is_dir():

        partes = list(
            caminho.rglob("*.parquet")
        )

        if not partes:
            raise RuntimeError(
                "Diretorio sem arquivos Parquet."
            )

        return sum(
            int(
                pq.read_metadata(str(p)).num_rows
            )
            for p in partes
        )

    raise FileNotFoundError(caminho)


def sim_ja_processado(item):

    controle = controle_sim(item)

    if not controle.exists():
        return False

    try:

        dados = json.loads(
            controle.read_text(
                encoding="utf-8"
            )
        )

        if (
            dados.get("status") != "CONCLUIDO"
            or dados.get("chave") != item["chave"]
        ):
            return False

        validar_parquet_sim(dados["destino"])

        return True

    except Exception:
        return False


def registrar_conclusao_sim(item, destino, linhas):

    salvar_json_atomico(
        controle_sim(item),
        {
            "execution_id": EXECUTION_ID,
            "fonte": FONTE,
            "grupo": GRUPO,
            "chave": item["chave"],
            "uf": item["uf"],
            "ano": int(item["ano"]),
            "abrangencia": item["abrangencia"],
            "arquivo_remoto": item["caminho"],
            "destino": str(destino),
            "linhas": int(linhas),
            "status": "CONCLUIDO",
            "processado_em": datetime.now(
                timezone.utc
            ).isoformat()
        }
    )


def registrar_log_sim(dados):

    with ARQUIVO_LOG.open(
        "a", encoding="utf-8"
    ) as arquivo:

        arquivo.write(
            json.dumps(
                dados,
                ensure_ascii=False,
                default=str
            ) + "\n"
        )


# ============================================================
# 02 - COPIAR PARQUET
# ============================================================

def copiar_parquet_sim(origem, destino):

    origem = Path(origem)
    destino = Path(destino)

    temporario = Path(
        str(destino) + ".partial"
    )

    if temporario.exists():

        if temporario.is_dir():
            shutil.rmtree(temporario)
        else:
            temporario.unlink()

    if origem.is_file():

        if origem.suffix.lower() != ".parquet":
            raise RuntimeError(
                "Formato nao Parquet: " + str(origem)
            )

        shutil.copy2(origem, temporario)

    elif origem.is_dir():

        if not list(
            origem.rglob("*.parquet")
        ):
            raise RuntimeError(
                "Origem sem Parquet."
            )

        shutil.copytree(origem, temporario)

    else:

        raise FileNotFoundError(origem)

    linhas = validar_parquet_sim(temporario)

    if destino.exists():

        if destino.is_dir():
            shutil.rmtree(destino)
        else:
            destino.unlink()

    temporario.replace(destino)

    return linhas


# ============================================================
# 03 - LOCALIZAR ARQUIVO REMOTO
# ============================================================

# O catálogo foi descoberto sem filtro de UF.
# O FileBag também será consultado sem filtro
# de UF e de grupo, evitando o problema anterior.

BAGS_POR_ANO = {}


def localizar_sim(item):

    ano = int(item["ano"])

    if ano not in BAGS_POR_ANO:

        bag = pysus.ftp.sim(
            year=ano,
            source=SOURCE,
            download=False
        )

        BAGS_POR_ANO[ano] = bag

    bag = BAGS_POR_ANO[ano]

    candidatos = list(bag.files)

    # Primeiro, comparar o caminho completo.
    for indice, remoto in enumerate(candidatos):

        caminho = str(
            getattr(remoto, "path", "") or ""
        )

        if caminho == item["caminho"]:
            return bag, indice

    # Fallback por nome somente quando unico.
    correspondencias = []

    for indice, remoto in enumerate(candidatos):

        nome = Path(
            str(getattr(remoto, "path", ""))
        ).name.upper()

        if nome == item["arquivo"].upper():
            correspondencias.append(indice)

    if len(correspondencias) == 1:

        return bag, correspondencias[0]

    raise RuntimeError(
        "Arquivo remoto nao encontrado "
        "de forma inequivoca: "
        + item["caminho"]
    )


# ============================================================
# 04 - CARREGAR CATALOGO
# ============================================================

if not ARQUIVO_CATALOGO.exists():

    raise RuntimeError(
        "Execute a Parte 2 primeiro."
    )

df_catalogo = pd.read_parquet(
    ARQUIVO_CATALOGO
)

df_catalogo = df_catalogo[
    df_catalogo["ano"].isin(ANOS)
].copy()

FILA_DOWNLOAD = (
    df_catalogo
    .sort_values(["ano", "uf", "arquivo"])
    .to_dict(orient="records")
)

if not FILA_DOWNLOAD:
    raise RuntimeError("Catalogo vazio.")

# Conferir falhas de descoberta.
checkpoint = json.loads(
    ARQUIVO_CHECKPOINT.read_text(
        encoding="utf-8"
    )
)

estado = checkpoint.get("consultas", {})

anos_com_erro = [
    ano for ano in ANOS
    if estado.get(str(ano), {}).get("status")
    not in ("CONCLUIDO", "SEM_ARQUIVO")
]

if anos_com_erro:
    raise RuntimeError(
        f"Consultas pendentes: {anos_com_erro}"
    )

# ============================================================
# 05 - PREPARAR PENDENTES
# ============================================================

pendentes = [
    item
    for item in FILA_DOWNLOAD
    if SOBRESCREVER or not sim_ja_processado(item)
]

ja_existentes = len(FILA_DOWNLOAD) - len(pendentes)

if MAX_ARQUIVOS_EXECUCAO is not None:
    pendentes = pendentes[:MAX_ARQUIVOS_EXECUCAO]

print("Catalogados:", len(FILA_DOWNLOAD))
print("Ja processados:", ja_existentes)
print("Pendentes nesta execucao:", len(pendentes))

# ============================================================
# 06 - DOWNLOAD
# ============================================================

concluidos = 0
erros = 0
novas_linhas = 0

if SOMENTE_CATALOGO:

    print("Modo somente catalogo: download desabilitado.")

else:

    for numero, item in enumerate(
        pendentes, start=1
    ):

        print("\n" + "-" * 65)

        print(
            f"[{numero}/{len(pendentes)}] "
            f"{item['uf']} - {item['ano']}"
        )

        print("Arquivo:", item["arquivo"])

        sucesso = False
        ultimo_erro = None

        for tentativa in range(
            1, MAX_TENTATIVAS + 1
        ):

            try:

                print(
                    f"Download tentativa {tentativa}"
                )

                bag, indice = localizar_sim(item)

                local = bag.download_one(indice)

                origem = Path(str(local.path))

                destino = destino_sim(item)

                linhas = copiar_parquet_sim(
                    origem, destino
                )

                registrar_conclusao_sim(
                    item, destino, linhas
                )

                registrar_log_sim({
                    "execution_id": EXECUTION_ID,
                    "chave": item["chave"],
                    "uf": item["uf"],
                    "ano": item["ano"],
                    "arquivo": item["arquivo"],
                    "linhas": linhas,
                    "status": "CONCLUIDO"
                })

                concluidos += 1
                novas_linhas += linhas
                sucesso = True

                print("CONCLUIDO:", destino)
                print("Registros:", linhas)

                break

            except Exception as erro:

                ultimo_erro = (
                    f"{type(erro).__name__}: {erro}"
                )

                print("ERRO:", ultimo_erro)

                # Descartar FileBag possivelmente
                # associado a uma conexao invalida.
                BAGS_POR_ANO.pop(
                    int(item["ano"]), None
                )

                if tentativa < MAX_TENTATIVAS:
                    time.sleep(
                        ESPERA_ENTRE_TENTATIVAS
                        * tentativa
                    )

        if not sucesso:

            erros += 1

            registrar_log_sim({
                "execution_id": EXECUTION_ID,
                "chave": item["chave"],
                "uf": item["uf"],
                "ano": item["ano"],
                "arquivo": item["arquivo"],
                "status": "ERRO",
                "erro": ultimo_erro
            })

        time.sleep(ESPERA_ENTRE_ARQUIVOS)

# ============================================================
# 07 - RESUMO
# ============================================================

print("\n" + "=" * 70)
print("INGESTAO SIM/DO FINALIZADA")
print("=" * 70)

print("Arquivos novos:", concluidos)
print("Ja existentes:", ja_existentes)
print("Erros:", erros)
print("Novos registros:", novas_linhas)
print("Execution ID:", EXECUTION_ID)

print("=" * 70)
print("PARTE 3 CONCLUIDA")